In [2]:

import numpy as np
import pandas as pd
import rioxarray as rxr
from osgeo import gdal
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import accuracy_score
from sklearn.metrics import mean_absolute_error, root_mean_squared_error
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline, make_pipeline
import geopandas as gpd
import rasterio
from shapely.geometry import box
import pandas as pd
from shapely.geometry import Point
from rasterio.sample import sample_gen


In [3]:

def save_img(output_filename, reference_filename, array):
    # 打开参考数据
    reference = gdal.Open(reference_filename)
    
    # 获取地理变换参数和坐标系
    geo_transform = reference.GetGeoTransform()
    projection = reference.GetProjection()

    # 获取矩阵尺寸
    rows, cols = array.shape

    # 创建新的 TIFF 数据集
    driver = gdal.GetDriverByName('GTiff')
    out_data = driver.Create(output_filename, cols, rows, 1, gdal.GDT_Float32)

    # 设置地理变换和坐标系
    out_data.SetGeoTransform(geo_transform)
    out_data.SetProjection(projection)
    

    # 写入数据
    out_data.GetRasterBand(1).WriteArray(array)

    # 保存并关闭文件
    out_data.FlushCache()
    out_data = None

In [9]:
location = 'wash_bay'
region = 'Region2'
input_bands = 'all_bands'

method = 'RF' 
merge_year = 23_24
year = 2024

ICEsat_point_path_2023 = f'G:\ICESat-2\{region}_map\{location}_map\A2023\A_{location}_30m_2023.shp'
ICEsat_point_path_2024 = f'G:\ICESat-2\{region}_map\{location}_map\A2024\A_{location}_30m_2024.shp'

point_2023 = gpd.read_file(ICEsat_point_path_2023)
point_2024 = gpd.read_file(ICEsat_point_path_2024)

point_2023 = point_2023.to_crs(point_2024.crs)

l = pd.DataFrame(point_2023)
ll = pd.DataFrame(point_2024)
z = pd.concat([l, ll])

print(z.shape)
data = z

(4597, 18)


In [10]:

#extract y_train and x_train data
if input_bands == 'all_bands':
    #target_band =  ['b1_Wash_20',  'b4_Wash_20', 'b5_Wash_20','b6_Wash_20', 'b7_Wash_20', 'b8_Wash_20']
    #['b1_Morecam', 'b4_Morecam','b5_Morecam', 'b6_Morecam', 'b7_Morecam', 'b8_Morecam'] 'b2_FB_',
    target_band = ['b1_FB6',  'b2_FB6','b3_FB6',  'b4_FB6' ]#,'b5_FB2', 'b6_FB2', 'b7_FB2']
    #target_band = ['b1_Thames_', 'b2_Thames_', 'b3_Thames_', 'b4_Thames_', 'b5_Thames_', 'b6_Thames_']#, 'b5_FB6', 'b6_FB6'] # 
    #target_band = ['b1_FB6','b6_FB6', 'b5_FB6', 'b6_FB6','b7_FB6', 'b8_FB0']
    #[f'b1_{location}_', f'b4_{location}_',f'b5_{location}_',f'b6_{location}_',f'b7_{location}_',f'b8_{location}_']
    target_id = [0, 1, 2, 3]#, 4, 5] #

    #output_path = f'G:\ICESat-2\Region1\Region1_Thames\A{year}\{location}_{year}_6_{method}_FB6.tif'
    output_path = f'G:\ICESat-2\{region}_map\{location}_map\A{year}\{location}_{year}_4_{method}_FB6_{merge_year}.tif'
    #output_path  = f'G:\Freq\{location}\{location}_{year}_6_bands_{method}.tif'
else:
    target_band = ['b1_FB6'] # ['b1_ribble_']# ['b1_Morecam']#[f'b1_{location}_']
    target_id = [0]    
    #output_path  = f'G:\ICESat-2\Region2\Wash_bay\A{year}\{location}_{year}_Freq_{method}zz_FB6.tif'
    output_path = f'F:\ICESat-2\{region}_map\{location}_map\A{year}\{location}_{year}_Freq_{method}_FB6.tif'


y_train = data['OH'].values
x_train = data[target_band].values
print('x_train:', x_train.shape)
print('y_train', y_train.shape)

#select calssifer
if method == 'RF':
    #classifier = RandomForestRegressor(n_estimators=300, random_state=30)
    classifier = RandomForestRegressor(n_estimators=300, min_samples_leaf=5, random_state=30)
    #classifier = RandomForestRegressor(n_estimators=300, max_depth=15, random_state=30)
    classifier.fit(x_train, y_train)
elif method == 'Linear':
    classifier = np.polyfit(x_train[:, 0], y_train, deg=1)
elif method == 'Poly':
    classifier = np.polyfit(x_train[:, 0], y_train, deg=3)
elif method == 'MpLinear':
    classifier = LinearRegression()
    classifier.fit(x_train, y_train)
elif method == 'MpPoly':
    degree = 3  # 设置多项式的阶数
    classifier = make_pipeline(PolynomialFeatures(degree), LinearRegression())
    classifier.fit(x_train, y_train)
    # classifier = PolynomialFeatures()
    # classifier.fit(x_train, y_train)
else:
    print('error classifier input')

# # load Ramdon Forest classifer
# rf_regressor = RandomForestRegressor(n_estimators=100, random_state=30)
# rf_regressor.fit(x_train, y_train)

#load raster 
#raster_path = 'G:\Freq\Ribble_Dee\Ribble_Dee_2022_FBs_30m_addL7.tif'
# raster_path = f'G:\Freq\{location}\{location}_{year}_FBs_30m_addL7.tif'
# #raster_path = 'G:\Freq\wash\Wash_2022_FB.tif'
# raster_path = f'G:\ICESat-2\Region1\Region1_Thames\A{year}\Thames_esturay_{year}_FB6.tif'

raster_path = f'G:\ICESat-2\{region}_map\{location}_map\A{year}\{location}_{year}_FB6.tif'
#raster_path = 'G:\ICESat-2\Region2\Wash_bay\A2022\Wash_2022_FBs_30m_addL7.tif'

raster = rxr.open_rasterio(raster_path).values
raster = np.nan_to_num(raster, nan = 0)
selected_raster = raster[target_id, :, :]

if len(selected_raster.shape) != 3:
    row, col = selected_raster.shape
    selected_raster = selected_raster.reshape(1, row, col)
else: 
    bands, row, col = selected_raster.shape

print('selected_raster.shape', selected_raster.shape)


x_train: (4597, 4)
y_train (4597,)
selected_raster.shape (4, 1535, 1656)


In [11]:
# Topograph mapping 

k = 0
result = np.array([[]])
for i in range(row):
    
    x_train = np.transpose(selected_raster[:, i, :])
    if method == 'RF':
        y_pred = classifier.predict(x_train)
    elif  method == 'Linear' :
        y_pred = np.polyval(classifier, x_train)
    elif  method == 'Poly' :
        y_pred = np.polyval(classifier, x_train)
    elif method == 'MpLinear':
        y_pred = classifier.predict(x_train)
    elif method == 'MpPoly':
        
        y_pred = classifier.predict(x_train)

    result = np.append(result, y_pred)
    if i%(int(row/5)) == 0:
        
        print(k*20 , '%_ is done')
        k = k+1

print(result.shape, type(result))

tf_mask = raster[6,:,:].flatten()
result[tf_mask == 0] = 0
result_ = result.reshape(row, col)

save_img(output_path, raster_path, result_)

0 %_ is done
20 %_ is done
40 %_ is done
60 %_ is done
80 %_ is done
(2541960,) <class 'numpy.ndarray'>
